# Technology — Capacity Planning Methodology
---

<div style="padding:15px;border-left:8px solid #fa709a;background:#fff0f5;border-radius:4px;">
<strong>Core Insight:</strong> Capacity planning is Sean's key differentiator. While most
data engineering candidates know SQL and Python, almost none have experience preventing
capacity outages through data-driven forecasting. This is a direct Citi story —
own it completely.
</div>

### The Four-Step Loop
Sean's actual methodology from Citi:
1. **Baseline** — 90-day rolling average from CA APM
2. **Model** — identify the capacity driver (requests/sec, not time)
3. **Forecast** — project the trend, set ceiling at 70% utilization
4. **Alert** — trigger when projected trend line hits the ceiling, not when current reading hits it

## 🧠 Why Capacity Planning is a Data Engineering Problem

| Step | The Data Engineering Skill |
|------|--------------------------|
| Baseline | Time-series aggregation — rolling averages, percentiles, seasonal decomposition |
| Model | Feature engineering — what DRIVES utilization? (traffic, batch jobs, user growth) |
| Forecast | Linear regression, trend extrapolation, confidence intervals |
| Alert | Threshold logic on the projected value, not current reading |

### The Ceiling Rule
**Never alert at 100%** — by then it's too late.
Alert at **70%** on the trend line, giving 6-8 weeks of lead time for procurement.
Server provisioning at Citi took 3-4 weeks (hardware + configuration + security scan).
A 70% ceiling on a linear trend gives 30pp headroom = 3-4 weeks of safe growth at typical rates.

### The Capacity Driver Rule
Don't model CPU vs time. Model CPU vs requests/second.
Time is a proxy — the real driver is load.
When a batch job moves from 2am to 11pm, a time-based model fails. A load-based model adapts.

In [ ]:
import numpy as np
import json
from datetime import date, timedelta

# ══════════════════════════════════════
# STEP 1: BASELINE — 90-day rolling average
# ══════════════════════════════════════

np.random.seed(42)

# Simulate 90 days of CPU readings for one server
days = np.arange(90)
# Normal utilization with slight upward trend
baseline_cpu = 35 + 0.15 * days + np.random.normal(0, 3, 90)
dates = [date(2024, 1, 1) + timedelta(days=int(d)) for d in days]

# 90-day statistics
p50 = np.percentile(baseline_cpu, 50)
p95 = np.percentile(baseline_cpu, 95)
rolling_7day = np.array([baseline_cpu[max(0,i-6):i+1].mean() for i in range(90)])

print("=== STEP 1: BASELINE ===")
print(f"90-day p50 CPU: {p50:.1f}%")
print(f"90-day p95 CPU: {p95:.1f}%")
print(f"Current (day 90): {baseline_cpu[-1]:.1f}%")
print(f"7-day rolling avg: {rolling_7day[-1]:.1f}%")
print(f"Trend: +{0.15:.2f}% per day")

In [ ]:
# ══════════════════════════════════════
# STEP 2 & 3: MODEL + FORECAST
# ══════════════════════════════════════

import numpy as np

np.random.seed(42)
days_history = np.arange(90)
baseline_cpu = 35 + 0.15 * days_history + np.random.normal(0, 3, 90)

# Fit a linear trend to the last 30 days (most recent trend matters most)
last_30 = baseline_cpu[-30:]
x = np.arange(30)

# Linear regression: y = mx + b
coeffs = np.polyfit(x, last_30, 1)
slope = coeffs[0]      # percentage points per day
intercept = coeffs[1]  # baseline

print("=== STEP 2: MODEL ===")
print(f"Linear trend: +{slope:.3f}% CPU per day")
print(f"At this rate: +{slope*7:.2f}% per week, +{slope*30:.2f}% per month")

# Project forward 60 days
CEILING = 70.0
future_days = np.arange(60)
projected_cpu = (intercept + 30 * slope) + slope * future_days  # from day 90 forward

# Find when we hit the ceiling
above_ceiling = np.where(projected_cpu >= CEILING)[0]
days_until_breach = above_ceiling[0] if len(above_ceiling) > 0 else None

print("
=== STEP 3: FORECAST ===")
print(f"Current CPU trend: {slope:.3f}%/day")
print(f"Ceiling:           {CEILING}%")
if days_until_breach is not None:
    print(f"Projected breach:  Day +{days_until_breach} ({days_until_breach} days from today)")
    from datetime import date, timedelta
    breach_date = date(2024, 4, 1) + timedelta(days=int(days_until_breach))
    print(f"Breach date:       {breach_date}")
    print(f"Lead time:         {days_until_breach} days (provisioning takes 21-28 days)")
    print(f"Action needed:     {'NOW — within lead time!' if days_until_breach < 35 else 'Monitor weekly'}")
else:
    print("No breach projected in next 60 days")

In [ ]:
# ══════════════════════════════════════
# STEP 4: ALERT — Proactive, not reactive
# ══════════════════════════════════════

import numpy as np
from datetime import date, timedelta

def capacity_alert(server_id: str, cpu_readings: np.ndarray,
                   ceiling: float = 70.0, lead_days: int = 35) -> dict:
    """
    Evaluate capacity risk for a server.

    Returns:
        alert: bool — should we act now?
        days_until_breach: int or None
        action: str — what to do
    """
    if len(cpu_readings) < 14:
        return {"alert": False, "action": "insufficient_data"}

    # Fit trend to last 14 days
    x = np.arange(len(cpu_readings))
    slope, intercept = np.polyfit(x, cpu_readings, 1)
    current = cpu_readings[-1]

    # Project forward
    days_until_breach = None
    if slope > 0:  # only breaches if trend is upward
        # current_value + slope * n_days = ceiling
        if current < ceiling:
            n = (ceiling - current) / slope
            days_until_breach = int(n)

    alert = (days_until_breach is not None and days_until_breach <= lead_days)
    action = "PROVISION NOW" if alert else ("MONITOR" if days_until_breach else "STABLE")

    return {
        "server_id": server_id,
        "current_cpu": round(float(current), 1),
        "slope_per_day": round(float(slope), 3),
        "days_until_breach": days_until_breach,
        "alert": alert,
        "action": action
    }

# Test on a problem server (fast-growing)
np.random.seed(0)
problem_server = 40 + np.arange(30) * 1.2 + np.random.normal(0, 2, 30)  # +1.2%/day
stable_server  = 55 + np.random.normal(0, 3, 30)  # stable at 55%

print("Problem server:", capacity_alert("SRV-9001", problem_server))
print("Stable server: ", capacity_alert("SRV-9002", stable_server))

## 📊 Fleet-Wide Capacity Dashboard

The capacity alert runs across all 6,000 servers daily:

```python
# Conceptual: daily capacity scan
alerts = []
for server_id in all_servers:
    readings = fetch_90day_cpu(server_id)
    result = capacity_alert(server_id, readings)
    if result["alert"]:
        alerts.append(result)

# Sort by urgency (fewest days until breach)
alerts.sort(key=lambda x: x["days_until_breach"])

# Report top 20 most urgent
print(f"CAPACITY ALERT REPORT — {date.today()}")
print(f"Servers at risk (breach within 35 days): {len(alerts)}")
for a in alerts[:20]:
    print(f"  {a['server_id']}: {a['days_until_breach']} days | {a['current_cpu']}% CPU | +{a['slope_per_day']}%/day")
```

### The Capacity Report Format (Citi Style)
```
SERVER: SRV-1042 (Production, App: TradingEngine)
Current CPU: 67%
Trend: +0.8% per day (+5.6% per week)
Projected breach (70%): Day +4 = 2024-04-05
Action: PROVISION NOW — lead time is 21 days, breach in 4 days
Owner: Infrastructure Team A (escalation: auto)
```

## 🎤 5 Interview Q&A

**Q1: What is capacity planning and how is it different from monitoring?**
Monitoring tells you what's happening NOW — reactive. Capacity planning tells you what
WILL happen and WHEN — proactive. Monitoring: "CPU is at 85% right now, alert!"
Capacity planning: "CPU is trending at +0.8%/day, will breach 70% in 4 days."
The goal of capacity planning is to never have a monitoring alert fire in production.

---

**Q2: Why 70% ceiling instead of 100%?**
Provisioning hardware takes time — at Citi, 3-4 weeks for hardware + config + security scan.
70% provides ~30 percentage points of headroom. At typical growth rates (+0.8%/day),
30pp = ~37 days of lead time. Setting the ceiling at 90% or 95% only gives 5-12 days —
not enough time to provision before impact.

---

**Q3: Why model against requests/second rather than time?**
Time is a proxy for load — the real driver. When a batch job moves from 2am to 6pm,
a time-based model predicts the wrong utilization for that time slot.
A load-based model (CPU vs requests/sec) adapts: if traffic doubles, the model says
CPU will double. This is more robust to schedule changes, traffic spikes, and application changes.

---

**Q4: How do you handle seasonality in capacity forecasting?**
Simple linear regression ignores day-of-week and time-of-day patterns.
For systems with strong seasonality (business-hours peaks, monthly billing spikes):
(1) Use STL decomposition to separate trend from seasonal patterns.
(2) Model trend separately from seasonal component.
(3) Forecast trend, then add back the seasonal pattern for the target time.
In practice, weekly seasonality (Mon-Fri vs weekend) is often enough to model.

---

**Q5: How do you communicate capacity risk to non-technical stakeholders?**
Translate to business impact: "We will run out of compute capacity for the TradingEngine
on April 5th. This will cause transaction processing delays during peak trading hours.
Provisioning new servers takes 3 weeks and costs $X. We need to order by March 15th."
Avoid: "CPU utilization will exceed the 70th percentile threshold in 4 days."
Use: "The trading engine will slow down in 4 days if we don't act by tomorrow."

## 📚 Key Terms

| Term | Definition |
|------|------------|
| **Capacity Driver** | The metric that best predicts resource utilization — usually load (requests/sec), not time |
| **Utilization Ceiling** | The threshold above which performance degrades — typically 70-80% for CPU/memory |
| **Lead Time** | Time from ordering to delivery — determines how far ahead the forecast must look |
| **Linear Regression** | `y = mx + b` — fit a line to historical data to project future values |
| **Trend** | The long-term direction of a metric (slope of the regression line) |
| **Seasonality** | Repeating patterns by time of day / week / month — separate from trend |
| **STL Decomposition** | Seasonal-Trend decomposition using LOESS — separates trend, seasonal, residual components |
| **p95 / p99** | The 95th/99th percentile — used for capacity sizing (don't size for average, size for peak) |
| **Proactive vs Reactive** | Proactive = act before the breach. Reactive = act after the outage. |
| **Provisioning** | Allocating, configuring, and deploying new compute resources |

## 💼 The Citi Narrative

**Context:** Citi APM infrastructure — 6,000 monitored endpoints, growing application portfolio.
The capacity team was getting paged at 3am when servers hit 100% utilization.

**The Problem:** Monitoring was reactive. No system existed to predict when servers would
breach their utilization ceiling. The team was always in firefighting mode.

**The Four-Step Solution:**
1. **Baseline:** Pulled 90-day CPU data from CA APM API into PostgreSQL via Python ETL
2. **Model:** Linear regression on last 30 days per server, slope = capacity driver
3. **Forecast:** Project trend forward 60 days, compare to 70% ceiling
4. **Alert:** Daily Jira ticket auto-created for any server breaching in < 35 days

**The Result:**
- Caught 3 servers on a collision course with capacity 8 weeks before breach
- Provisioned new hardware before impact — zero downtime
- Alert volume for reactive capacity issues: dropped from 15/month to 0 in the following 12 months

**The Number:** 47 days — the first server flagged was 47 days from breach when the system
caught it. Provisioning took 28 days. Margin: 19 days. Plenty.

**Interview Line:** *"I was tired of getting paged at 3am for something we could have seen
coming 6 weeks earlier. So I built a forecasting model — 90-day baseline, linear trend,
70% ceiling alert. The first time it fired, we had 47 days of lead time. We provisioned
the server, the breach never happened, and I slept through the night it would have paged us."*

In [ ]:
# Practice: Extend the capacity alert function

import numpy as np
from datetime import date, timedelta

def capacity_report(servers_data: dict, ceiling: float = 70.0) -> list:
    """
    Generate a capacity planning report for a fleet of servers.

    Args:
        servers_data: dict of server_id -> array of daily CPU readings (90 days)
        ceiling: utilization ceiling (default 70%)

    Returns:
        List of alert dicts, sorted by urgency (fewest days to breach first)
    """
    # Your implementation:
    # 1. For each server, fit a linear trend to last 30 days
    # 2. Project when it hits the ceiling
    # 3. Return servers breaching within 35 days, sorted by urgency

    alerts = []
    for server_id, readings in servers_data.items():
        readings = np.array(readings)
        last30 = readings[-30:]
        x = np.arange(30)
        slope, intercept = np.polyfit(x, last30, 1)
        current = readings[-1]

        days_to_breach = None
        if slope > 0 and current < ceiling:
            days_to_breach = int((ceiling - current) / slope)

        if days_to_breach is not None and days_to_breach <= 35:
            alerts.append({
                "server_id": server_id,
                "current_cpu": round(float(current), 1),
                "days_to_breach": days_to_breach,
                "breach_date": str(date.today() + timedelta(days=days_to_breach))
            })

    return sorted(alerts, key=lambda x: x["days_to_breach"])

# Test it
np.random.seed(42)
fleet = {
    "SRV-001": 30 + np.arange(90)*0.9 + np.random.normal(0,2,90),   # fast growing
    "SRV-002": 55 + np.random.normal(0,3,90),                         # stable
    "SRV-003": 45 + np.arange(90)*0.4 + np.random.normal(0,2,90),   # slow growing
}
report = capacity_report(fleet)
print(f"Servers at risk: {len(report)}")
for a in report:
    print(f"  {a['server_id']}: breach in {a['days_to_breach']} days ({a['breach_date']})")

## 🎯 Summary

### The Four-Step Loop
1. **Baseline** → 90-day rolling data, p50/p95 per server
2. **Model** → Linear regression on last 30 days, slope = trend
3. **Forecast** → Project to 70% ceiling, compute days until breach
4. **Alert** → Jira/Slack when breach within 35 days (lead time = 28 days)

### Interview Confidence Checklist
- [ ] Can explain the four-step loop from memory
- [ ] Can explain why 70% ceiling, not 100%
- [ ] Can explain why model load (requests/sec) not time
- [ ] Can write the linear regression + threshold logic in Python
- [ ] Can name the Citi story: 3 servers caught 8 weeks early, zero downtime

**"Simplicity and clarity is Gold."** — Sean's Study Mantra 🚀